## Part I: From Probability to Language

**Question 1: Why is this decomposition useful for generating text?**
The autoregressive decomposition is useful because it breaks the complex joint probability of an entire sentence into a sequence of simpler, conditional probabilities. This perfectly mirrors the left-to-right process of generating text step-by-step, allowing a model to predict the next token based entirely on the history of tokens generated so far.

## Part II: A Bayesian Network for Text

**Question 2: What independence assumption is being made by this network?**
The first-order Markov language model assumes that the current token depends only on the immediately preceding token, entirely independent of all prior history.
In probability notation: $P(X_t | X_1, X_2, \dots, X_{t-1}) = P(X_t | X_{t-1})$.

In [4]:
import random
from collections import defaultdict

# Part III: Build a Small Language Dataset
corpus = [
    "<START> the cat sat on the mat <END>",
    "<START> the cat sat on the rug <END>",
    "<START> the dog sat on the mat <END>",
    "<START> the dog ran to the park <END>",
    "<START> the cat ran to the park <END>",
    "<START> the dog sat on the rug <END>"
]

# Tokenize
sentences = [sentence.split() for sentence in corpus]

# Part IV & V: Constructing the First-Order Model
transition_counts = defaultdict(lambda: defaultdict(int))

# Count transitions
for sentence in sentences:
    for i in range(len(sentence) - 1):
        current_word = sentence[i]
        next_word = sentence[i+1]
        transition_counts[current_word][next_word] += 1

# Construct Conditional Probability Table P(X_t | X_t-1)
cpt_1st_order = defaultdict(dict)
for current_word, next_words in transition_counts.items():
    total_transitions = sum(next_words.values())
    for next_word, count in next_words.items():
        cpt_1st_order[current_word][next_word] = count / total_transitions

## Part III & IV: Dataset and Constructing the CPT

**Question 3: Construct the conditional probability distribution for specific words.**
Based on the dataset provided, the conditional probabilities $P(\text{next word} \mid \text{current word})$ are:
*   **the**: `{'cat': 0.5, 'dog': 0.5, 'mat': 0.333, 'rug': 0.333, 'park': 0.333}` 
*   **cat**: `{'sat': 0.667, 'ran': 0.333}`
*   **dog**: `{'sat': 0.667, 'ran': 0.333}`
*   **sat**: `{'on': 1.0}`
*   **ran**: `{'to': 1.0}`
*   **Zero-probability transitions**: Any combination not observed in the text (e.g., $P(\text{dog} \mid \text{cat}) = 0$, $P(\text{sat} \mid \text{sat}) = 0$) has a zero probability.

## Part V & VI: Inspect the Code

**Question 4: Where in the program are the transition counts stored?**
The transition counts are stored in the nested dictionary `transition_counts`, which maps the `current_word` to a dictionary of `next_word` frequencies.

**Question 5: Where is $P(X_t\vert{}X_{t-1})$ computed?**
It is computed in the loop where `count / total_transitions` is evaluated and assigned to the `cpt_1st_order[current_word][next_word]` dictionary.

**Question 6: How does the program choose the next word?**
Depending on the generation mode implemented later, it can either always choose the highest probability (greedy) or sample proportionally to the distribution (sampling). Sampling respects the true probabilistic nature of the model by utilizing random choice weighted by $P(X_t\vert{}X_{t-1})$.

**Question 7: What happens if the program encounters a word for which no transition has been observed?**
A `KeyError` will be raised when attempting to sample, or the model will halt unexpectedly, because the dictionary does not contain a probability distribution for an unobserved state.

In [5]:
# Part VII: Probability Normalization Test
print("Probability Normalization Test:")
for word in cpt_1st_order:
    total = sum(cpt_1st_order[word].values())
    print(f"{word}: {total:.4f}")

Probability Normalization Test:
<START>: 1.0000
the: 1.0000
cat: 1.0000
sat: 1.0000
on: 1.0000
mat: 1.0000
rug: 1.0000
dog: 1.0000
ran: 1.0000
to: 1.0000
park: 1.0000


## Part VII: Test the Probability Model

**Question 8: If one of the totals is 0.87, what does this tell you about the implementation?**
If a total sums to 0.87 instead of 1.0, it indicates a bug in the implementation. The model is either failing to count all possible transitions for that context, improperly calculating the denominator, or losing precision. For a valid probability distribution, $\sum_{v}P(v|w)$ must strictly equal 1.

In [7]:
# Part VIII: Predicting next word
def predict_most_probable(cpt, context):
    if context not in cpt:
        return None
    return max(cpt[context], key=cpt[context].get)

print(f"\nMost probable after 'the': {predict_most_probable(cpt_1st_order, 'the')}")
print(f"Most probable after 'cat': {predict_most_probable(cpt_1st_order, 'cat')}")

# Part X: Deterministic vs Probabilistic Generation
# ADDED max_length=20 to prevent infinite loops
def generate_sentence(cpt, mode="sampling", order=1, max_length=20):
    sentence = ["<START>"]
    if order == 2:
        sentence = ["<START>", "<START>"] 
        
    # The loop now checks if the sentence has exceeded the max length
    while len(sentence) < max_length: 
        if order == 1:
            context = sentence[-1]
        else:
            context = (sentence[-2], sentence[-1])
            
        if context not in cpt:
            break
            
        if mode == "greedy":
            next_word = max(cpt[context], key=cpt[context].get)
        else: 
            words = list(cpt[context].keys())
            probs = list(cpt[context].values())
            next_word = random.choices(words, weights=probs)[0]
            
        sentence.append(next_word)
        if next_word == "<END>":
            break
            
    return " ".join(sentence)

print("\nGreedy Generation (5 examples):")
for _ in range(5):
    print(generate_sentence(cpt_1st_order, mode="greedy"))

print("\nSampling Generation (5 examples):")
for _ in range(5):
    print(generate_sentence(cpt_1st_order, mode="sampling"))


Most probable after 'the': cat
Most probable after 'cat': sat

Greedy Generation (5 examples):
<START> the cat sat on the cat sat on the cat sat on the cat sat on the cat sat
<START> the cat sat on the cat sat on the cat sat on the cat sat on the cat sat
<START> the cat sat on the cat sat on the cat sat on the cat sat on the cat sat
<START> the cat sat on the cat sat on the cat sat on the cat sat on the cat sat
<START> the cat sat on the cat sat on the cat sat on the cat sat on the cat sat

Sampling Generation (5 examples):
<START> the cat ran to the mat <END>
<START> the park <END>
<START> the park <END>
<START> the mat <END>
<START> the mat <END>


## Part VIII, IX & X: Predicting and Generating Text

**Question 9: Are the most probable predictions always the same as the words that you would personally expect?**
No. Because the model strictly relies on a tiny frequency distribution, it lacks common-sense reasoning and semantic understanding. A probability model strictly reflects mathematical frequencies of the training text, whereas human linguistic expectations draw on vast real-world context and grammar.

**Question 10: Compare the two sets of generated sentences. Which mode produces more variation? Why?**
Sampling produces significantly more variation. Greedy generation repeatedly selects `arg max_w P(w|w_{previous})`, leading to identical deterministic output every time it starts with the same token. Sampling generates text by picking words probabilistically based on their weights, allowing it to explore different valid paths in the Bayesian network.

## Part XII: Use the LLM Again (Testing the Second-Order Model)

Below is the LLM-generated code for the second-order autoregressive model. To test that the generated implementation correctly matches the probabilistic specification, we sample 5 sentences using $P(X_t|X_{t-2}, X_{t-1})$.

In [8]:
# Part XII: Second-Order Model Implementation
transition_counts_2nd = defaultdict(lambda: defaultdict(int))

for sentence in sentences:
    for i in range(len(sentence) - 2):
        context = (sentence[i], sentence[i+1])
        next_word = sentence[i+2]
        transition_counts_2nd[context][next_word] += 1

cpt_2nd_order = defaultdict(dict)
for context, next_words in transition_counts_2nd.items():
    total_transitions = sum(next_words.values())
    for next_word, count in next_words.items():
        cpt_2nd_order[context][next_word] = count / total_transitions

print("\nSecond-Order Sampling Generation (5 examples):")
for _ in range(5):
    prime_sentence = ["<START>", "the"]
    while True:
        context = (prime_sentence[-2], prime_sentence[-1])
        if context not in cpt_2nd_order:
            break
        words = list(cpt_2nd_order[context].keys())
        probs = list(cpt_2nd_order[context].values())
        next_word = random.choices(words, weights=probs)[0]
        prime_sentence.append(next_word)
        if next_word == "<END>":
            break
    print(" ".join(prime_sentence))


Second-Order Sampling Generation (5 examples):
<START> the dog ran to the park <END>
<START> the cat sat on the rug <END>
<START> the cat sat on the rug <END>
<START> the dog ran to the park <END>
<START> the dog sat on the mat <END>


## Part XI, XII & XIII: Second-Order Bayesian Network

**Question 11: How does the second-order model differ from the first-order model?**
1.  **Graph Structure:** Nodes now have two parents ($X_{t-2} \rightarrow X_t \leftarrow X_{t-1}$) instead of one.
2.  **CPT:** The probability table conditions on a tuple of two words $P(X_t | X_{t-2}, X_{t-1})$ rather than a single word.
3.  **Context:** It has a larger window (2 preceding words), allowing it to capture slightly longer-term dependencies.
4.  **Data Needs:** It requires exponentially more data to reliably estimate probabilities because the state space of possible word pairs is vastly larger than single words.

**Question 12: Why does increasing the amount of context potentially improve prediction? Why can it simultaneously make the model harder to estimate from limited data?**
Increasing context improves prediction because it restricts ambiguity (e.g., "to the" strongly implies a location like "park", whereas "the" alone could precede a subject or object). It makes the model harder to estimate because the size of the CPT grows exponentially. In limited data, most word-pair contexts will have zero observations, leading to sparsity and an inability to predict the next word when encountering unseen combinations.

## Part XV: Reflection and Conceptual Conclusions

**Question 13: Why is Approach B preferable when constructing an intelligent system?**
Approach B is preferable because it explicitly defines the intended behavioral and probabilistic invariants rather than relying on a black-box generation. By specifying the mathematical model (e.g., $P(X_t|X_{t-1})$) and testing its invariants (like probabilities summing to 1), engineers can clearly distinguish the conceptual model from the code implementation, ensuring reliability and making debugging structural rather than guesswork.

**Question 14: What did thinking of the language model as a Bayesian network give you?**
Viewing the language model as a Bayesian network provided several key benefits:
*   **A factorisation of the joint distribution:** It mathematically justified breaking the complex sequence of words into computable, step-by-step conditional probabilities via the chain rule.
*   **A principled method for generation:** It outlined how generation is directly tied to traversing the network sequentially—sampling from the conditional probability table at each node.
*   **A way to reason about independence assumptions:** It made clear exactly what information was being discarded (e.g., everything before $X_{t-1}$ in a first-order model) and allowed for structured analysis of how increasing the context window changed the network structure.

---
### Reflection on LLM Usage
*During this lab, prompting an LLM to generate code using Approach B yielded an implementation explicitly tailored to conditional probability tables rather than defaulting to modern neural libraries. However, inspecting the LLM-generated code was crucial. For example, ensuring that the normalization step properly divided by the sum of occurrences to satisfy $\sum_{v}P(v|w)=1$ required careful validation. Validating these probabilistic invariants guaranteed that the underlying logic functioned as a true probability distribution and not just an arbitrary frequency counter.*